In [ ]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 79.6 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


## Local Inference on GPU
Model page: https://huggingface.co/distilbert/distilbert-base-uncased

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/distilbert/distilbert-base-uncased)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("fill-mask", model="distilbert/distilbert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForMaskedLM

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")
model = AutoModelForMaskedLM.from_pretrained("distilbert/distilbert-base-uncased")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

## Remote Inference via Inference Providers
Ensure you have a valid **HF_TOKEN** set in your environment. You can get your token from [your settings page](https://huggingface.co/settings/tokens). Note: running this may incur charges above the free tier.
The following Python example shows how to run the model remotely on HF Inference Providers, automatically selecting an available inference provider for you.
For more information on how to use the Inference Providers, please refer to our [documentation and guides](https://huggingface.co/docs/inference-providers/en/index).

In [12]:
import os
from google.colab import userdata

# Securely fetch the token from Colab Secrets
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN set successfully.')
except userdata.SecretNotFoundError:
    print('Please add your HF_TOKEN to the Colab Secrets (key icon on the left).')

HF_TOKEN set successfully.


In [14]:
import os
from huggingface_hub import InferenceClient

# Ensure the token exists before initializing the client
token = os.getenv('HF_TOKEN')

if token:
    client = InferenceClient(
        provider="auto",
        api_key=token,
    )

    result = client.fill_mask(
        "The answer to the universe is [MASK].",
        model="distilbert/distilbert-base-uncased",
    )
    display(result)
else:
    print('HF_TOKEN is missing. Please check your Colab Secrets.')

[FillMaskOutputElement(score=0.15373733639717102, sequence='the answer to the universe is unknown.', token=4242, token_str='unknown', fill_mask_output_token_str=None),
 FillMaskOutputElement(score=0.019274476915597916, sequence='the answer to the universe is entropy.', token=23077, token_str='entropy', fill_mask_output_token_str=None),
 FillMaskOutputElement(score=0.01653728447854519, sequence='the answer to the universe is infinite.', token=10709, token_str='infinite', fill_mask_output_token_str=None),
 FillMaskOutputElement(score=0.015316938050091267, sequence='the answer to the universe is zero.', token=5717, token_str='zero', fill_mask_output_token_str=None),
 FillMaskOutputElement(score=0.013508927077054977, sequence='the answer to the universe is infinity.', token=15579, token_str='infinity', fill_mask_output_token_str=None)]

In [15]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("jp797498e/twitter-entity-sentiment-analysis")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'twitter-entity-sentiment-analysis' dataset.
Path to dataset files: /kaggle/input/twitter-entity-sentiment-analysis


In [16]:
import os

# Tentukan path lengkap ke file CSV
# Sesuaikan nama file jika berbeda setelah cek output Cell 2
PATH_DATASET = os.path.join(path, 'twitter_training.csv')

# Verifikasi file ada
if os.path.exists(PATH_DATASET):
    print(f"Dataset ditemukan di: {PATH_DATASET}")
else:
    # Jika nama file berbeda, cari otomatis
    csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
    print(f"File CSV yang ditemukan: {csv_files}")
    if csv_files:
        PATH_DATASET = os.path.join(path, csv_files[0])
        print(f"Menggunakan file: {PATH_DATASET}")
    else:
        print("ERROR: Tidak ada file CSV ditemukan!")

Dataset ditemukan di: /kaggle/input/twitter-entity-sentiment-analysis/twitter_training.csv


In [17]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    confusion_matrix
)

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline
)

# Cek device (GPU atau CPU)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device yang digunakan : {device}")
print(f"Versi PyTorch         : {torch.__version__}")

if device == 'cuda':
    print(f"Nama GPU              : {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM tersedia         : {vram:.1f} GB")
else:
    print("GPU tidak tersedia, menggunakan CPU (training akan lebih lambat)")

Device yang digunakan : cuda
Versi PyTorch         : 2.10.0+cu128
Nama GPU              : Tesla T4
VRAM tersedia         : 15.6 GB


In [18]:
# Load dataset CSV
# Dataset tidak punya header, jadi kita beri nama kolom manual
df = pd.read_csv(
    PATH_DATASET,
    names    = ['tweet_id', 'entity', 'sentiment', 'text'],
    encoding = 'utf-8'
)

print("=" * 50)
print("INFO DATASET")
print("=" * 50)
print(f"Jumlah total baris : {len(df):,}")
print(f"Jumlah kolom       : {len(df.columns)}")
print(f"Nama kolom         : {list(df.columns)}")

print("\n=== DISTRIBUSI SENTIMEN ===")
for sentimen, count in df['sentiment'].value_counts().items():
    persen = count / len(df) * 100
    print(f"  {sentimen:<12}: {count:>6,} baris ({persen:.1f}%)")

print("\n=== CONTOH ISI DATA ===")
pd.set_option('display.max_colwidth', 70)
print(df[['sentiment', 'text']].sample(5, random_state=42).to_string(index=False))

INFO DATASET
Jumlah total baris : 74,682
Jumlah kolom       : 4
Nama kolom         : ['tweet_id', 'entity', 'sentiment', 'text']

=== DISTRIBUSI SENTIMEN ===
  Negative    : 22,542 baris (30.2%)
  Positive    : 20,832 baris (27.9%)
  Neutral     : 18,318 baris (24.5%)
  Irrelevant  : 12,990 baris (17.4%)

=== CONTOH ISI DATA ===
 sentiment                                                                                                                                                                                                                                                 text
Irrelevant                                                                                                                                                                            He said told u I'm getting in that box of a brain dead controller player.
  Positive                                                                                                                                                   

In [19]:
def bersihkan_teks(teks):
    """
    Membersihkan teks tweet dari noise sebelum dikirim ke model BERT.

    Yang dihapus:
    - URL (http/https/www)
    - Mention pengguna (@username)
    - Simbol hashtag (#) tapi bukan kata-katanya
    - Karakter non-ASCII (emoji, karakter unicode aneh)
    - Karakter selain huruf, angka, dan tanda baca dasar
    - Spasi berlebih

    Yang TIDAK dihapus:
    - Tanda baca seperti ! ? . , karena BERT bisa manfaatkan ini
    - Kapitalisasi huruf (BERT uncased akan lowercase sendiri)
    """
    if pd.isna(teks):
        return ""

    teks = str(teks)

    # Hapus URL
    teks = re.sub(r'http\S+|www\S+|https\S+', '', teks)

    # Hapus mention (@username)
    teks = re.sub(r'@\w+', '', teks)

    # Hapus simbol # tapi pertahankan kata setelahnya
    teks = re.sub(r'#', '', teks)

    # Hapus karakter non-ASCII
    teks = re.sub(r'[^\x00-\x7F]+', '', teks)

    # Hapus karakter selain huruf, angka, spasi, tanda baca dasar
    teks = re.sub(r"[^a-zA-Z0-9\s.,!?'\"-]", '', teks)

    # Hapus spasi berlebih
    teks = re.sub(r'\s+', ' ', teks).strip()

    return teks


# Mapping label teks ke angka (model hanya mengerti angka)
label_mapping = {
    'Positive'   : 2,
    'Neutral'    : 1,
    'Negative'   : 0,
    'Irrelevant' : 1   # Irrelevant diperlakukan sama seperti Neutral
}

# Nama kelas untuk laporan evaluasi nanti
label_nama = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}

print("Memproses dataset, harap tunggu...")

df['text_bersih'] = df['text'].apply(bersihkan_teks)
df['label']       = df['sentiment'].map(label_mapping)

# Hapus baris yang labelnya tidak dikenali atau teks kosong
df = df.dropna(subset=['text_bersih', 'label'])
df = df[df['text_bersih'].str.len() > 5]
df['label'] = df['label'].astype(int)

print(f"Dataset setelah preprocessing: {len(df):,} baris")

print("\n=== DISTRIBUSI LABEL SETELAH MAPPING ===")
for lid, count in df['label'].value_counts().sort_index().items():
    print(f"  {lid} ({label_nama[lid]}): {count:,} baris")

print("\n=== CONTOH SEBELUM vs SESUDAH PREPROCESSING ===")
for _, row in df.sample(3, random_state=1).iterrows():
    print(f"  ASLI   : {str(row['text'])[:80]}")
    print(f"  BERSIH : {row['text_bersih'][:80]}")
    print(f"  LABEL  : {row['sentiment']} → {row['label']}")
    print()

Memproses dataset, harap tunggu...
Dataset setelah preprocessing: 71,670 baris

=== DISTRIBUSI LABEL SETELAH MAPPING ===
  0 (Negative): 21,684 baris
  1 (Neutral): 30,145 baris
  2 (Positive): 19,841 baris

=== CONTOH SEBELUM vs SESUDAH PREPROCESSING ===
  ASLI   : and An amazing read aloud book for you and your child! Select on Amazon The Wish
  BERSIH : and An amazing read aloud book for you and your child! Select on Amazon The Wish
  LABEL  : Neutral → 1

  ASLI   : BLACK OPS COLD WAR ZOMBIES DIFFERENT FICKING INSANE. youtu.be / hT6SnyPugz4
  BERSIH : BLACK OPS COLD WAR ZOMBIES DIFFERENT FICKING INSANE. youtu.be hT6SnyPugz4
  LABEL  : Positive → 2

  ASLI   : Seriously you bro what the damned FUCK have y ’ all u been doing @Xbox
  BERSIH : Seriously you bro what the damned FUCK have y all u been doing
  LABEL  : Irrelevant → 1



In [20]:
# Jumlah sampel yang digunakan untuk training
# Rekomendasi:
#   - CPU          : 2000  (training ~15-30 menit)
#   - GPU T4 Colab : 6000  (training ~5-10 menit)
#   - GPU A100     : 15000 (training ~5-10 menit, akurasi lebih tinggi)
SAMPLE_SIZE = 6000 if device == 'cuda' else 2000

df_sample = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=42)

X = df_sample['text_bersih'].tolist()
y = df_sample['label'].tolist()

# Split 80% training, 20% testing
# stratify=y memastikan proporsi label sama di keduanya
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size  = 0.2,
    random_state = 42,
    stratify   = y
)

print(f"Total sampel digunakan : {len(df_sample):,}")
print(f"Data training          : {len(X_train):,} baris (80%)")
print(f"Data testing           : {len(X_test):,} baris (20%)")

print(f"\nDistribusi label di training set:")
for lid, count in sorted(Counter(y_train).items()):
    print(f"  {lid} ({label_nama[lid]}): {count:,}")

print(f"\nDistribusi label di testing set:")
for lid, count in sorted(Counter(y_test).items()):
    print(f"  {lid} ({label_nama[lid]}): {count:,}")

Total sampel digunakan : 6,000
Data training          : 4,800 baris (80%)
Data testing           : 1,200 baris (20%)

Distribusi label di training set:
  0 (Negative): 1,482
  1 (Neutral): 2,008
  2 (Positive): 1,310

Distribusi label di testing set:
  0 (Negative): 371
  1 (Neutral): 502
  2 (Positive): 327


In [21]:
# Nama model yang digunakan
# distilbert-base-uncased:
#   - Versi ringan dari BERT (40% lebih kecil, 60% lebih cepat)
#   - Dilatih pada teks bahasa Inggris
#   - "uncased" = tidak membedakan huruf besar/kecil
#   - Ukuran download: ~260 MB (hanya sekali, lalu di-cache)
MODEL_NAME = "distilbert/distilbert-base-uncased"

print(f"Memuat tokenizer dari Hugging Face Hub...")
print(f"Model : {MODEL_NAME}")
print("(Jika pertama kali, akan download ~260 MB — harap tunggu...)\n")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Tokenizer berhasil dimuat!")

print("\nMemuat arsitektur model...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels             = 3,     # 3 kelas output: Negative, Neutral, Positive
    ignore_mismatched_sizes = True  # reset classification head untuk 3 kelas
)

# Pindahkan model ke GPU jika tersedia
model = model.to(device)

print("Model berhasil dimuat!")
print(f"\nDetail model:")
total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Total parameter    : {total_params:,}")
print(f"  Parameter trainable: {trainable:,}")
print(f"  Device             : {next(model.parameters()).device}")

Memuat tokenizer dari Hugging Face Hub...
Model : distilbert/distilbert-base-uncased
(Jika pertama kali, akan download ~260 MB — harap tunggu...)

Tokenizer berhasil dimuat!

Memuat arsitektur model...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model berhasil dimuat!

Detail model:
  Total parameter    : 66,955,779
  Parameter trainable: 66,955,779
  Device             : cuda:0


In [22]:
print("Melakukan tokenisasi data training...")
train_encodings = tokenizer(
    X_train,
    truncation    = True,
    padding       = True,
    max_length    = 128,      # cukup untuk tweet (biasanya < 50 token)
    return_tensors = 'pt'     # kembalikan sebagai PyTorch tensor
)

print("Melakukan tokenisasi data testing...")
test_encodings = tokenizer(
    X_test,
    truncation    = True,
    padding       = True,
    max_length    = 128,
    return_tensors = 'pt'
)

print("Tokenisasi selesai!")
print(f"\nShape input_ids training : {train_encodings['input_ids'].shape}")
print(f"Shape input_ids testing  : {test_encodings['input_ids'].shape}")
# Contoh output: torch.Size([4800, 128]) → 4800 sampel, 128 token per sampel


class SentimentDataset(torch.utils.data.Dataset):
    """
    Custom Dataset class untuk Hugging Face Trainer.

    Trainer membutuhkan objek Dataset yang:
    1. Bisa diakses per index: dataset[0], dataset[1], dst.
    2. Punya method __len__() untuk tahu total data
    3. Setiap item berisi 'input_ids', 'attention_mask', dan 'labels'
    """
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = torch.tensor(labels, dtype=torch.long)

    def __getitem__(self, idx):
        # Ambil semua field tokenisasi untuk index ke-idx
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.labels)


train_dataset = SentimentDataset(train_encodings, y_train)
test_dataset  = SentimentDataset(test_encodings,  y_test)

print(f"\nObjek Dataset berhasil dibuat:")
print(f"  train_dataset : {len(train_dataset):,} sampel")
print(f"  test_dataset  : {len(test_dataset):,} sampel")

Melakukan tokenisasi data training...
Melakukan tokenisasi data testing...
Tokenisasi selesai!

Shape input_ids training : torch.Size([4800, 128])
Shape input_ids testing  : torch.Size([1200, 128])

Objek Dataset berhasil dibuat:
  train_dataset : 4,800 sampel
  test_dataset  : 1,200 sampel


In [23]:
import numpy as np

def hitung_metrik(eval_pred):
    """
    Callback yang dipanggil Trainer setiap akhir epoch untuk evaluasi.
    Menghitung akurasi prediksi terhadap label asli.
    """
    logits, labels = eval_pred
    predictions    = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, predictions)}


# Konfigurasi proses training
training_args = TrainingArguments(
    output_dir                  = './hasil_training',
    num_train_epochs            = 3,
    per_device_train_batch_size = 16,    # kurangi ke 8 jika muncul error VRAM
    per_device_eval_batch_size  = 32,
    warmup_ratio                = 0.1,   # 10% langkah awal untuk warmup LR
    weight_decay                = 0.01,  # regularisasi L2 ringan
    learning_rate               = 2e-5,  # learning rate standar untuk fine-tuning BERT
    logging_steps               = 50,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "accuracy",
    report_to                   = "none",
    fp16                        = (device == 'cuda'),  # half-precision hanya di GPU
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = test_dataset,
    compute_metrics = hitung_metrik,
)

print("=" * 55)
print("MEMULAI FINE-TUNING MODEL")
print("=" * 55)
print(f"  Model     : {MODEL_NAME}")
print(f"  Device    : {device}")
print(f"  Epoch     : {training_args.num_train_epochs}")
print(f"  Batch     : {training_args.per_device_train_batch_size}")
print(f"  LR        : {training_args.learning_rate}")
print(f"  Training  : {len(train_dataset):,} sampel")
print(f"  Testing   : {len(test_dataset):,} sampel")
print("=" * 55)
print()

trainer.train()

print("\nTraining selesai!")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


MEMULAI FINE-TUNING MODEL
  Model     : distilbert/distilbert-base-uncased
  Device    : cuda
  Epoch     : 3
  Batch     : 16
  LR        : 2e-05
  Training  : 4,800 sampel
  Testing   : 1,200 sampel



Epoch,Training Loss,Validation Loss,Accuracy
1,0.863990,0.823931,0.612500
2,0.687718,0.797358,0.635000
3,0.501118,0.815668,0.631667


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



Training selesai!


In [24]:
print("=" * 55)
print("EVALUASI FINAL MODEL PADA DATA TEST")
print("=" * 55)

# Jalankan prediksi pada seluruh test set
output       = trainer.predict(test_dataset)
y_pred       = np.argmax(output.predictions, axis=-1)
y_true       = np.array(y_test)
nama_kelas   = ['Negative', 'Neutral', 'Positive']

# Akurasi keseluruhan
akurasi = accuracy_score(y_true, y_pred)
print(f"\nAkurasi keseluruhan : {akurasi:.4f}  ({akurasi*100:.2f}%)")

# Laporan lengkap per kelas
print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_true, y_pred, target_names=nama_kelas))

# Penjelasan metrik:
print("Penjelasan metrik:")
print("  Precision : dari semua prediksi kelas X, berapa % yang benar")
print("  Recall    : dari semua data asli kelas X, berapa % yang terdeteksi")
print("  F1-Score  : rata-rata harmonis precision dan recall (metrik utama)")
print("  Support   : jumlah sampel asli per kelas di test set")

# Confusion Matrix yang rapi
print("\n=== CONFUSION MATRIX ===")
cm = confusion_matrix(y_true, y_pred)
header = f"{'':14}" + "".join(f"{n:>12}" for n in nama_kelas)
print(header)
print("-" * (14 + 12 * len(nama_kelas)))
for i, row in enumerate(cm):
    print(f"Asli {nama_kelas[i]:<9}", end="")
    for val in row:
        print(f"{val:>12}", end="")
    print()

print("\nDiagonal = prediksi BENAR | Luar diagonal = prediksi SALAH")

EVALUASI FINAL MODEL PADA DATA TEST



Akurasi keseluruhan : 0.6358  (63.58%)

=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

    Negative       0.64      0.68      0.66       371
     Neutral       0.62      0.64      0.63       502
    Positive       0.66      0.57      0.61       327

    accuracy                           0.64      1200
   macro avg       0.64      0.63      0.63      1200
weighted avg       0.64      0.64      0.64      1200

Penjelasan metrik:
  Precision : dari semua prediksi kelas X, berapa % yang benar
  Recall    : dari semua data asli kelas X, berapa % yang terdeteksi
  F1-Score  : rata-rata harmonis precision dan recall (metrik utama)
  Support   : jumlah sampel asli per kelas di test set

=== CONFUSION MATRIX ===
                  Negative     Neutral    Positive
--------------------------------------------------
Asli Negative          253          93          25
Asli Neutral           107         323          72
Asli Positive           38         102     

In [25]:
# Mount Google Drive terlebih dahulu jika belum
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# Path penyimpanan model di Drive
PATH_SAVE = '/content/drive/MyDrive/sentiment_project/model_finetuned'
os.makedirs(PATH_SAVE, exist_ok=True)

# Simpan model dan tokenizer
trainer.save_model(PATH_SAVE)
tokenizer.save_pretrained(PATH_SAVE)

print(f"Model berhasil disimpan ke Google Drive!")
print(f"Lokasi: {PATH_SAVE}")
print(f"\nFile yang tersimpan:")
for f in os.listdir(PATH_SAVE):
    ukuran = os.path.getsize(os.path.join(PATH_SAVE, f)) / 1e6
    print(f"  {f:<35} ({ukuran:.1f} MB)")

Mounted at /content/drive


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model berhasil disimpan ke Google Drive!
Lokasi: /content/drive/MyDrive/sentiment_project/model_finetuned

File yang tersimpan:
  config.json                         (0.0 MB)
  model.safetensors                   (267.8 MB)
  training_args.bin                   (0.0 MB)
  tokenizer_config.json               (0.0 MB)
  tokenizer.json                      (0.7 MB)


In [26]:
# Buat pipeline prediksi dari model yang sudah ditraining
sentiment_pipeline = pipeline(
    task      = "text-classification",
    model     = model,
    tokenizer = tokenizer,
    device    = 0 if device == 'cuda' else -1
)

# Mapping ID label ke nama yang mudah dibaca
id_ke_sentimen = {
    'LABEL_0': 'Negative',
    'LABEL_1': 'Neutral',
    'LABEL_2': 'Positive'
}
emoji_map = {
    'Negative': '😠 NEGATIVE',
    'Neutral' : '😐 NEUTRAL',
    'Positive': '😊 POSITIVE'
}


def prediksi_sentimen(input_teks):
    """
    Fungsi prediksi sentimen untuk teks baru.

    Parameter:
        input_teks (str atau list of str): tweet yang ingin dianalisis

    Return:
        list of dict berisi 'teks', 'sentimen', 'confidence'
    """
    # Pastikan input berupa list
    if isinstance(input_teks, str):
        input_teks = [input_teks]

    # Preprocessing (harus sama persis dengan saat training)
    teks_bersih = [bersihkan_teks(t) for t in input_teks]

    # Prediksi
    hasil_raw = sentiment_pipeline(
        teks_bersih,
        truncation = True,
        max_length = 128
    )

    # Format hasil yang mudah dibaca
    hasil = []
    for teks_asli, raw in zip(input_teks, hasil_raw):
        sentimen   = id_ke_sentimen[raw['label']]
        confidence = raw['score']
        hasil.append({
            'teks'       : teks_asli[:100] + ('...' if len(teks_asli) > 100 else ''),
            'sentimen'   : sentimen,
            'label_display': emoji_map[sentimen],
            'confidence' : f"{confidence * 100:.1f}%"
        })

    return hasil


# ==============================
# UJI COBA PREDIKSI
# ==============================
tweet_uji = [
    "I absolutely love this new feature! Works perfectly!",
    "The package arrived today. It was in a brown box.",
    "Terrible experience. The product broke after one day. Never again.",
    "Just watched the game. It was okay I guess.",
    "BEST DAY EVER!! Got promoted and it's my birthday!!",
    "Service has been down for 3 hours and no response from support.",
]

print("=" * 60)
print("HASIL PREDIKSI SENTIMEN")
print("=" * 60)

hasil_uji = prediksi_sentimen(tweet_uji)
for i, h in enumerate(hasil_uji, 1):
    print(f"\n[{i}] {h['teks']}")
    print(f"    Hasil      : {h['label_display']}")
    print(f"    Confidence : {h['confidence']}")

HASIL PREDIKSI SENTIMEN

[1] I absolutely love this new feature! Works perfectly!
    Hasil      : 😊 POSITIVE
    Confidence : 85.7%

[2] The package arrived today. It was in a brown box.
    Hasil      : 😐 NEUTRAL
    Confidence : 41.7%

[3] Terrible experience. The product broke after one day. Never again.
    Hasil      : 😠 NEGATIVE
    Confidence : 71.2%

[4] Just watched the game. It was okay I guess.
    Hasil      : 😊 POSITIVE
    Confidence : 65.1%

[5] BEST DAY EVER!! Got promoted and it's my birthday!!
    Hasil      : 😊 POSITIVE
    Confidence : 63.8%

[6] Service has been down for 3 hours and no response from support.
    Hasil      : 😠 NEGATIVE
    Confidence : 82.1%


In [29]:
print("=" * 55)
print("MODE PREDIKSI INTERAKTIF")
print("Ketik tweet → tekan Enter → lihat hasil sentimen")
print("Ketik 'selesai' untuk berhenti")
print("=" * 55)

while True:
    print()
    tweet_input = input("Masukkan tweet: ").strip()

    if tweet_input.lower() in ['selesai', 'keluar', 'exit', 'quit', 'q']:
        print("\nSesi prediksi selesai. Terima kasih!")
        break

    if len(tweet_input) < 3:
        print("Teks terlalu pendek, coba lagi.")
        continue

    hasil = prediksi_sentimen(tweet_input)
    h = hasil[0]
    print(f"  Sentimen   : {h['label_display']}")
    print(f"  Confidence : {h['confidence']}")

MODE PREDIKSI INTERAKTIF
Ketik tweet → tekan Enter → lihat hasil sentimen
Ketik 'selesai' untuk berhenti

Masukkan tweet: This is life-changing! I can't believe how much time I saved using this app.
  Sentimen   : 😊 POSITIVE
  Confidence : 62.4%

Masukkan tweet: The movie is 2 hours long and starts at 7 PM tonight.
  Sentimen   : 😐 NEUTRAL
  Confidence : 62.5%

Masukkan tweet: Extremely disappointed. The UI is confusing and the app keeps crashing on my phone.
  Sentimen   : 😠 NEGATIVE
  Confidence : 90.4%

Masukkan tweet: Oh great, another update that breaks everything. Just what I needed.
  Sentimen   : 😠 NEGATIVE
  Confidence : 78.9%

Masukkan tweet: I've been waiting for my order for two weeks and still no update. This is unacceptable.


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  Sentimen   : 😠 NEGATIVE
  Confidence : 87.6%

Masukkan tweet: I'm so hyped for the new season! The trailer looks breathtaking.
  Sentimen   : 😊 POSITIVE
  Confidence : 85.4%

Masukkan tweet: selesai

Sesi prediksi selesai. Terima kasih!
